# 🎓 PROYECTO FINAL: Sistema Completo de Meta-Learning## Tu Propio Sistema Few-Shot End-to-End**¡Bienvenido al proyecto final!** Este es el momento de integrar todo lo aprendido.### 🎯 Objetivos del Proyecto:1. ✅ **Preparar datos reales**: Cargar y procesar Omniglot2. ✅ **Implementar algoritmos**: Prototypical Networks + opcional (Matching/MAML)3. ✅ **Entrenar correctamente**: Protocolo episódico end-to-end4. ✅ **Evaluar rigurosamente**: Métricas estándar con intervalos de confianza5. ✅ **Analizar resultados**: Comparar con baselines de papers6. ✅ **Experimentos avanzados**: Ablations, visualizaciones, OOD7. ✅ **Documentar**: Guardar modelos y resultados### 📊 Escenarios de Evaluación:| Escenario | Descripción | Dificultad ||-----------|-------------|------------|| 5-way 1-shot | 5 clases, 1 ejemplo cada una | ⭐⭐⭐ || 5-way 5-shot | 5 clases, 5 ejemplos cada una | ⭐⭐ || 20-way 1-shot | 20 clases, 1 ejemplo cada una | ⭐⭐⭐⭐⭐ |### 🏆 Criterios de Éxito (Omniglot):<table><tr>    <td><b>Escenario</b></td>    <td><b>Baseline (Random)</b></td>    <td><b>Bueno</b></td>    <td><b>Muy Bueno</b></td>    <td><b>Excelente</b></td></tr><tr>    <td>5-way 1-shot</td>    <td>20%</td>    <td>>85%</td>    <td>>92%</td>    <td>>97%</td></tr><tr>    <td>5-way 5-shot</td>    <td>20%</td>    <td>>95%</td>    <td>>97%</td>    <td>>99%</td></tr><tr>    <td>20-way 1-shot</td>    <td>5%</td>    <td>>70%</td>    <td>>80%</td>    <td>>90%</td></tr></table>### 📚 Lo que Aprenderás:- **Flujo completo**: De datos crudos a modelo en producción- **Best practices**: Código limpio, evaluación rigurosa- **Debugging**: Identificar y resolver problemas comunes- **Análisis**: Interpretar resultados y tomar decisiones**Tiempo estimado**: 3-4 horas---

## 📑 Table of Contents- [1 - Project Overview and Setup](#1)    - [1.1 - Learning Path](#1-1)    - [1.2 - Environment Setup](#1-2)- [2 - Data Preparation](#2)    - [2.1 - Loading Omniglot](#2-1)    - [2.2 - Creating Samplers](#2-2)    - [2.3 - Data Verification](#2-3)- [3 - Model Implementation](#3)    - [3.1 - Prototypical Networks](#3-1)    - [3.2 - Optional: Second Algorithm](#3-2)- [4 - Training Pipeline](#4)    - [4.1 - Training Loop](#4-1)    - [4.2 - Monitoring](#4-2)- [5 - Visualization](#5)- [6 - Evaluation](#6)    - [6.1 - Standard Evaluation](#6-1)    - [6.2 - Multiple Scenarios](#6-2)- [7 - Analysis and Comparison](#7)    - [7.1 - Compare with Papers](#7-1)    - [7.2 - Error Analysis](#7-2)- [8 - Advanced Experiments](#8)    - [8.1 - Hyperparameter Sensitivity](#8-1)    - [8.2 - Embedding Visualization](#8-2)    - [8.3 - OOD Robustness](#8-3)- [9 - Deployment](#9)    - [9.1 - Save Model](#9-1)    - [9.2 - Inference Pipeline](#9-2)- [10 - Reflection and Next Steps](#10)

<a name='1'></a>## 1 - Project Overview and Setup<a name='1-1'></a>### 1.1 - Learning PathThis project synthesizes **all 8 tutorials**:**From Tutorials 01-02**: Meta-learning fundamentals- Why few-shot learning?- Sample efficiency metrics**From Tutorial 03**: Prototypical Networks- Embedding space + prototypes- Euclidean distance classification**From Tutorial 03b**: Real datasets- Omniglot loading and preprocessing- N-way K-shot episodic sampling**From Tutorial 03c**: Matching Networks (optional)- Attention-based classification- Cosine similarity**From Tutorial 04**: MAML (optional)- Meta-optimization- Gradient-based adaptation**From Tutorial 08**: OOD Generalization- Robustness evaluation- Distribution shift handling**Your Mission:**Build a **production-ready** few-shot learning system that:1. Matches paper results (within 2-5%)2. Is well-documented and reproducible3. Can be deployed to new datasets easily**Success Metrics:**✅ **Functionality**: All components work✅ **Performance**: Within range of paper results✅ **Code Quality**: Clean, commented, modular✅ **Analysis**: Thoughtful interpretation of results

<a name='1-2'></a>### 1.2 - Environment Setup

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimimport torchvisionimport torchvision.transforms as transformsfrom torch.utils.data import Dataset, DataLoader, Subsetimport numpy as npimport matplotlib.pyplot as pltfrom matplotlib.patches import Rectanglefrom sklearn.manifold import TSNEfrom tqdm import tqdmimport jsonimport osfrom datetime import datetimefrom collections import defaultdictimport syssys.path.append('..')from utils.test_utils import print_success, print_hint, HintSystemfrom utils.data_utils import set_seedset_seed(42)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print("="*80)print(" " * 20 + "🚀 PROYECTO FINAL: META-LEARNING 🚀")print("="*80)print(f"\nDispositivo: {device}")print(f"PyTorch versión: {torch.__version__}")print(f"CUDA disponible: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")print("✅ Setup completo! ¡Comenzamos!\n")

<a name='2'></a>## 2 - Data Preparation<a name='2-1'></a>### 2.1 - Loading Omniglot**TODO 1**: Load Omniglot dataset with appropriate transforms.**Requirements:**- Background set for training (964 classes)- Evaluation set for testing (659 classes)- Resize to 28x28- Convert to tensor- Normalize (optional but recommended)**Hints:**- Use `torchvision.datasets.Omniglot`- `background=True` for train, `False` for test- Apply `transforms.Compose([...])`

In [ ]:
# TODO 1: Load Omniglot# Define transformstransform = transforms.Compose([    transforms.Resize((28, 28)),    transforms.ToTensor(),])# Load datasetsomniglot_train = torchvision.datasets.Omniglot(    root='../datasets',    background=True,    download=True,    transform=transform)omniglot_test = torchvision.datasets.Omniglot(    root='../datasets',    background=False,    download=True,    transform=transform)print(f"✅ Omniglot loaded successfully!")print(f"   Training classes: 964 (background)")print(f"   Test classes: 659 (evaluation)")print(f"   Total train images: {len(omniglot_train)}")print(f"   Total test images: {len(omniglot_test)}")print(f"   Image shape: {omniglot_train[0][0].shape}")# Verify datasample_img, sample_label = omniglot_train[0]print(f"\n   Sample image: {sample_img.shape}, label: {sample_label}")# Hintshints_data = HintSystem([    "Use torchvision.datasets.Omniglot with background=True/False",    "Transforms: Resize((28,28)), ToTensor()",    "Download will take 1-2 minutes first time",])hints_data.show_hint()

<a name='2-2'></a>### 2.2 - Creating N-Way K-Shot Sampler**TODO 2**: Implement episodic sampler for Omniglot.**Algorithm:**1. Organize dataset by class2. For each episode:   - Sample N classes randomly   - For each class, sample K support + Q query examples   - Assign new labels 0 to N-13. Return support_x, support_y, query_x, query_y

In [ ]:
# TODO 2: Implement samplerfrom collections import defaultdictimport randomclass OmniglotNWayKShot:    """    Episodic sampler for Omniglot.    Creates N-way K-shot episodes with Q query examples per class.    """    def __init__(self, dataset, n_way=5, k_shot=1, q_query=15):        self.dataset = dataset        self.n_way = n_way        self.k_shot = k_shot        self.q_query = q_query        # Organize dataset by class        self.class_to_indices = self._organize_by_class()        self.classes = list(self.class_to_indices.keys())        print(f"\n✅ Sampler initialized:")        print(f"   {self.n_way}-way {self.k_shot}-shot")        print(f"   {self.q_query} query per class")        print(f"   Total classes available: {len(self.classes)}")        print(f"   Support size: {self.n_way * self.k_shot}")        print(f"   Query size: {self.n_way * self.q_query}")    def _organize_by_class(self):        """Group image indices by class label."""        class_to_indices = defaultdict(list)        for idx in range(len(self.dataset)):            _, label = self.dataset[idx]            class_to_indices[label].append(idx)        return class_to_indices    def sample_episode(self):        """        Sample one episode.        Returns:            support_x: [n_way * k_shot, C, H, W]            support_y: [n_way * k_shot]            query_x: [n_way * q_query, C, H, W]            query_y: [n_way * q_query]        """        # Sample N classes        selected_classes = random.sample(self.classes, self.n_way)        support_x, support_y = [], []        query_x, query_y = [], []        for new_label, class_id in enumerate(selected_classes):            # Get all indices for this class            class_indices = self.class_to_indices[class_id]            # Sample K+Q examples            selected = random.sample(class_indices, self.k_shot + self.q_query)            # Split into support and query            support_indices = selected[:self.k_shot]            query_indices = selected[self.k_shot:]            # Add support examples            for idx in support_indices:                img, _ = self.dataset[idx]                support_x.append(img)                support_y.append(new_label)            # Add query examples            for idx in query_indices:                img, _ = self.dataset[idx]                query_x.append(img)                query_y.append(new_label)        # Convert to tensors        support_x = torch.stack(support_x)        support_y = torch.LongTensor(support_y)        query_x = torch.stack(query_x)        query_y = torch.LongTensor(query_y)        return support_x, support_y, query_x, query_y# Test the samplerprint("\n🧪 Testing sampler...")train_sampler = OmniglotNWayKShot(omniglot_train, n_way=5, k_shot=1, q_query=15)support_x, support_y, query_x, query_y = train_sampler.sample_episode()print(f"\n✅ Episode sampled successfully!")print(f"   Support: {support_x.shape}, labels: {support_y.shape}")print(f"   Query: {query_x.shape}, labels: {query_y.shape}")print(f"   Unique support labels: {torch.unique(support_y).tolist()}")print(f"   Unique query labels: {torch.unique(query_y).tolist()}")assert support_x.shape[0] == 5 * 1, "Support size mismatch"assert query_x.shape[0] == 5 * 15, "Query size mismatch"assert len(torch.unique(support_y)) == 5, "Should have 5 classes"print("   All assertions passed! ✅")

<a name='2-3'></a>### 2.3 - Data VerificationVisualize a sampled episode to verify everything works.

In [ ]:
# Visualize episodefig, axes = plt.subplots(2, 10, figsize=(20, 5))# Show support set (5 images)for i in range(5):    ax = axes[0, i]    ax.imshow(support_x[i].squeeze(), cmap='gray')    ax.set_title(f'Support\nClass {support_y[i].item()}', fontsize=10)    ax.axis('off')# Show first 10 query examplesfor i in range(10):    ax = axes[1, i]    if i < len(query_x):        ax.imshow(query_x[i].squeeze(), cmap='gray')        ax.set_title(f'Query\nClass {query_y[i].item()}', fontsize=10)    ax.axis('off')plt.suptitle('Example Episode: 5-way 1-shot', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("📊 Episode structure:")print(f"  • Top row: Support set (1 example per class)")print(f"  • Bottom row: Query set (first 10 of 75 total)")print(f"  • Model learns from support, predicts on query")

<a name='3'></a>## 3 - Model Implementation<a name='3-1'></a>### 3.1 - Prototypical Networks**TODO 3**: Implement Prototypical Networks.**Algorithm:**1. Embed all images: $z = f_\theta(x)$2. Compute prototypes: $c_k = \frac{1}{|S_k|} \sum_{(x,y) \in S_k} f_\theta(x)$3. Classify by distance: $p(y=k|x) \propto \exp(-d(f_\theta(x), c_k))$**Architecture:**- 4 convolutional blocks- Each block: Conv → BatchNorm → ReLU → MaxPool- Final embedding dimension: 64

In [ ]:
# TODO 3: Implement Prototypical Networksclass PrototypicalNetwork(nn.Module):    """    Prototypical Networks for Few-Shot Learning.    Reference: Prototypical Networks for Few-shot Learning (Snell et al., 2017)    """    def __init__(self, input_channels=1, embedding_dim=64):        super(PrototypicalNetwork, self).__init__()        # CNN encoder: 4 conv blocks        self.encoder = nn.Sequential(            # Block 1            nn.Conv2d(input_channels, 64, 3, padding=1),            nn.BatchNorm2d(64),            nn.ReLU(inplace=True),            nn.MaxPool2d(2),            # Block 2            nn.Conv2d(64, 64, 3, padding=1),            nn.BatchNorm2d(64),            nn.ReLU(inplace=True),            nn.MaxPool2d(2),            # Block 3            nn.Conv2d(64, 64, 3, padding=1),            nn.BatchNorm2d(64),            nn.ReLU(inplace=True),            nn.MaxPool2d(2),            # Block 4            nn.Conv2d(64, embedding_dim, 3, padding=1),            nn.BatchNorm2d(embedding_dim),            nn.ReLU(inplace=True),            nn.AdaptiveAvgPool2d(1)  # Global average pooling        )    def forward(self, support_x, support_y, query_x, n_way):        """        Forward pass for episode.        Args:            support_x: [n_way * k_shot, C, H, W]            support_y: [n_way * k_shot]            query_x: [n_queries, C, H, W]            n_way: number of classes        Returns:            logits: [n_queries, n_way]        """        # Embed all images        support_embeddings = self.encoder(support_x).squeeze()  # [n_support, emb_dim]        query_embeddings = self.encoder(query_x).squeeze()      # [n_query, emb_dim]        # Compute prototypes (class centroids)        prototypes = []        for k in range(n_way):            # Find support examples for class k            class_mask = (support_y == k)            class_embeddings = support_embeddings[class_mask]            # Compute mean            prototype = class_embeddings.mean(dim=0)            prototypes.append(prototype)        prototypes = torch.stack(prototypes)  # [n_way, emb_dim]        # Compute distances from queries to prototypes        # Using Euclidean distance        distances = torch.cdist(query_embeddings, prototypes)  # [n_query, n_way]        # Convert distances to logits (negative distance)        logits = -distances        return logits# Initialize modelmodel = PrototypicalNetwork(input_channels=1, embedding_dim=64).to(device)print("🧠 Prototypical Network initialized!")print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")# Test forward passmodel.eval()with torch.no_grad():    logits = model(        support_x.to(device),        support_y.to(device),        query_x[:10].to(device),        n_way=5    )print(f"\n✅ Forward pass test successful!")print(f"   Input: {support_x.shape} support, {query_x[:10].shape} query")print(f"   Output logits: {logits.shape}")print(f"   Logit range: [{logits.min():.2f}, {logits.max():.2f}]")

<a name='4'></a>## 4 - Training Pipeline<a name='4-1'></a>### 4.1 - Training Loop**TODO 4**: Implement episodic training loop.**Pseudocode:**```for episode in range(n_episodes):    1. Sample episode (support, query)    2. Forward pass → get logits    3. Compute cross-entropy loss    4. Backward pass    5. Optimize    6. Track metrics```**Tips:**- Use Adam optimizer (lr=0.001 is good default)- Track both loss and accuracy- Use tqdm for progress bar- Print every 100 episodes

In [ ]:
# TODO 4: Training loopdef train_prototypical(    model,    sampler,    n_episodes=2000,    lr=0.001,    print_every=100):    """    Train Prototypical Network with episodic training.    Args:        model: PrototypicalNetwork        sampler: OmniglotNWayKShot        n_episodes: number of training episodes        lr: learning rate        print_every: print stats every N episodes    Returns:        history: dict with 'loss' and 'accuracy' lists    """    model.train()    optimizer = optim.Adam(model.parameters(), lr=lr)    criterion = nn.CrossEntropyLoss()    history = {'loss': [], 'accuracy': []}    pbar = tqdm(range(n_episodes), desc="Training")    for episode in pbar:        # Sample episode        support_x, support_y, query_x, query_y = sampler.sample_episode()        # Move to device        support_x = support_x.to(device)        support_y = support_y.to(device)        query_x = query_x.to(device)        query_y = query_y.to(device)        # Forward pass        logits = model(support_x, support_y, query_x, n_way=sampler.n_way)        # Compute loss        loss = criterion(logits, query_y)        # Backward pass        optimizer.zero_grad()        loss.backward()        optimizer.step()        # Compute accuracy        preds = logits.argmax(dim=1)        accuracy = (preds == query_y).float().mean().item()        # Track metrics        history['loss'].append(loss.item())        history['accuracy'].append(accuracy)        # Update progress bar        pbar.set_postfix({            'loss': f'{loss.item():.3f}',            'acc': f'{accuracy:.3f}'        })        # Print detailed stats        if (episode + 1) % print_every == 0:            recent_loss = np.mean(history['loss'][-print_every:])            recent_acc = np.mean(history['accuracy'][-print_every:])            print(f"\n  Episode {episode+1}/{n_episodes}")            print(f"    Avg Loss: {recent_loss:.4f}")            print(f"    Avg Accuracy: {recent_acc:.4f} ({recent_acc*100:.2f}%)")    return history# Train the modelprint("\n" + "="*80)print(" " * 25 + "🚀 STARTING TRAINING")print("="*80 + "\n")train_sampler = OmniglotNWayKShot(omniglot_train, n_way=5, k_shot=1, q_query=15)model = PrototypicalNetwork().to(device)history = train_prototypical(    model,    train_sampler,    n_episodes=2000,    lr=0.001,    print_every=200)print("\n" + "="*80)print(" " * 20 + "✅ TRAINING COMPLETED!")print("="*80)print(f"\nFinal training accuracy: {history['accuracy'][-1]:.4f} ({history['accuracy'][-1]*100:.2f}%)")print(f"Final training loss: {history['loss'][-1]:.4f}")

<a name='5'></a>## 5 - Training VisualizationVisualize learning curves to understand training dynamics.

In [ ]:
# Plot training curvesfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))# Smooth curves with moving averagewindow = 50def smooth(values, window):    return np.convolve(values, np.ones(window)/window, mode='valid')# Loss curveax1.plot(history['loss'], alpha=0.3, color='steelblue', label='Raw')if len(history['loss']) > window:    ax1.plot(range(window-1, len(history['loss'])),             smooth(history['loss'], window),             color='steelblue', linewidth=2, label=f'Smoothed (window={window})')ax1.set_xlabel('Episode', fontsize=12)ax1.set_ylabel('Loss', fontsize=12)ax1.set_title('Training Loss', fontsize=14, fontweight='bold')ax1.legend()ax1.grid(True, alpha=0.3)# Accuracy curveax2.plot(history['accuracy'], alpha=0.3, color='forestgreen', label='Raw')if len(history['accuracy']) > window:    ax2.plot(range(window-1, len(history['accuracy'])),             smooth(history['accuracy'], window),             color='forestgreen', linewidth=2, label=f'Smoothed (window={window})')ax2.axhline(y=0.2, color='red', linestyle='--', alpha=0.5, label='Random (5-way)')ax2.set_xlabel('Episode', fontsize=12)ax2.set_ylabel('Accuracy', fontsize=12)ax2.set_title('Training Accuracy', fontsize=14, fontweight='bold')ax2.legend()ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("📊 Training Analysis:")print(f"  • Initial accuracy: {history['accuracy'][0]:.3f}")print(f"  • Final accuracy: {history['accuracy'][-1]:.3f}")print(f"  • Improvement: +{(history['accuracy'][-1] - history['accuracy'][0])*100:.1f}%")print(f"  • Final loss: {history['loss'][-1]:.4f}")# Check convergencelast_100 = history['accuracy'][-100:]variance = np.var(last_100)print(f"  • Variance (last 100 episodes): {variance:.6f}")if variance < 0.001:    print("  ✅ Model appears to have converged")else:    print("  ⚠️  Model may benefit from more training")

<a name='6'></a>## 6 - Rigorous Evaluation<a name='6-1'></a>### 6.1 - Standard Evaluation Protocol**TODO 5**: Implement evaluation with confidence intervals.**Protocol:**- Evaluate on 600 episodes (standard in papers)- Compute mean accuracy- Compute 95% confidence interval- Use TEST set (not training set!)

In [ ]:
# TODO 5: Evaluationdef evaluate_prototypical(    model,    sampler,    n_episodes=600):    """    Evaluate model following standard protocol.    Args:        model: trained PrototypicalNetwork        sampler: OmniglotNWayKShot on test set        n_episodes: number of test episodes    Returns:        mean_acc: mean accuracy        ci95: 95% confidence interval        accuracies: list of per-episode accuracies    """    model.eval()    accuracies = []    with torch.no_grad():        for _ in tqdm(range(n_episodes), desc="Evaluating"):            # Sample episode            support_x, support_y, query_x, query_y = sampler.sample_episode()            # Move to device            support_x = support_x.to(device)            support_y = support_y.to(device)            query_x = query_x.to(device)            query_y = query_y.to(device)            # Forward pass            logits = model(support_x, support_y, query_x, n_way=sampler.n_way)            # Compute accuracy            preds = logits.argmax(dim=1)            accuracy = (preds == query_y).float().mean().item()            accuracies.append(accuracy)    # Compute statistics    mean_acc = np.mean(accuracies)    std_acc = np.std(accuracies)    ci95 = 1.96 * std_acc / np.sqrt(n_episodes)    return mean_acc, ci95, accuracies# Evaluate on multiple scenariosprint("\n" + "="*80)print(" " * 22 + "📊 EVALUATION RESULTS")print("="*80 + "\n")scenarios = [    {'n_way': 5, 'k_shot': 1, 'q_query': 15, 'name': '5-way 1-shot'},    {'n_way': 5, 'k_shot': 5, 'q_query': 15, 'name': '5-way 5-shot'},    {'n_way': 20, 'k_shot': 1, 'q_query': 5, 'name': '20-way 1-shot'},]results = {}for scenario in scenarios:    print(f"\nEvaluating: {scenario['name']}")    print("-" * 60)    # Create sampler for this scenario    test_sampler = OmniglotNWayKShot(        omniglot_test,        n_way=scenario['n_way'],        k_shot=scenario['k_shot'],        q_query=scenario['q_query']    )    # Evaluate    mean_acc, ci95, accs = evaluate_prototypical(model, test_sampler, n_episodes=600)    # Store results    results[scenario['name']] = {        'mean': mean_acc,        'ci95': ci95,        'accuracies': accs    }    # Print results    print(f"\n✅ Results:")    print(f"   Accuracy: {mean_acc*100:.2f}% ± {ci95*100:.2f}%")    print(f"   95% CI: [{(mean_acc-ci95)*100:.2f}%, {(mean_acc+ci95)*100:.2f}%]")print("\n" + "="*80)

<a name='7'></a>## 7 - Analysis and Comparison<a name='7-1'></a>### 7.1 - Compare with Paper ResultsCompare your results with published benchmarks.

In [ ]:
# Compare with paper resultspaper_results = {    '5-way 1-shot': {'Prototypical': 98.8, 'Matching': 98.1, 'MAML': 98.7},    '5-way 5-shot': {'Prototypical': 99.7, 'Matching': 98.9, 'MAML': 99.9},    '20-way 1-shot': {'Prototypical': 96.0, 'Matching': 95.8, 'MAML': 95.8},}print("\n" + "="*80)print(" " * 18 + "📊 COMPARISON WITH PAPER RESULTS")print("="*80 + "\n")print(f"{'Scenario':<20} {'Your Result':>15} {'Paper (Proto)':>15} {'Difference':>15}")print("-" * 80)for scenario_name in ['5-way 1-shot', '5-way 5-shot', '20-way 1-shot']:    if scenario_name in results:        your_acc = results[scenario_name]['mean'] * 100        your_ci = results[scenario_name]['ci95'] * 100        paper_acc = paper_results[scenario_name]['Prototypical']        diff = your_acc - paper_acc        print(f"{scenario_name:<20} {your_acc:>6.2f}% ± {your_ci:>4.2f}% "              f"{paper_acc:>11.2f}% {diff:>12+.2f}%")print("\n💡 Interpretation:")print("  • Within ±2%: Excellent reproduction!")print("  • Within ±5%: Good, within experimental variance")print("  • More than ±5%: Check implementation or train longer")# Overall assessmentavg_diff = np.mean([    results[s]['mean'] * 100 - paper_results[s]['Prototypical']    for s in ['5-way 1-shot', '5-way 5-shot', '20-way 1-shot']    if s in results])print(f"\n📊 Average difference from paper: {avg_diff:+.2f}%")if abs(avg_diff) <= 2:    print("  🏆 EXCELLENT! Your implementation matches paper results!")elif abs(avg_diff) <= 5:    print("  ✅ GOOD! Results are within acceptable range.")else:    print("  ⚠️  Results differ significantly. Consider:")    print("     - Training longer")    print("     - Checking implementation details")    print("     - Adjusting hyperparameters")

<a name='9'></a>## 9 - Model Deployment<a name='9-1'></a>### 9.1 - Save Model and ResultsSave your trained model and results for future use.

In [ ]:
# Save modelmodel_path = 'prototypical_network_final.pth'torch.save({    'model_state_dict': model.state_dict(),    'training_history': history,    'evaluation_results': results,    'config': {        'embedding_dim': 64,        'training_episodes': 2000,        'learning_rate': 0.001,    }}, model_path)print(f"✅ Model saved to: {model_path}")# Save results to JSONresults_json = {    'timestamp': datetime.now().isoformat(),    'device': str(device),    'pytorch_version': torch.__version__,    'results': {        scenario: {            'accuracy_mean': float(data['mean']),            'accuracy_ci95': float(data['ci95']),        }        for scenario, data in results.items()    },    'training': {        'final_loss': float(history['loss'][-1]),        'final_accuracy': float(history['accuracy'][-1]),        'episodes': len(history['loss'])    }}results_path = 'results_final.json'with open(results_path, 'w') as f:    json.dump(results_json, f, indent=2)print(f"✅ Results saved to: {results_path}")print("\n📦 Saved artifacts:")print(f"  • {model_path} ({os.path.getsize(model_path) / 1024:.1f} KB)")print(f"  • {results_path} ({os.path.getsize(results_path) / 1024:.1f} KB)")

<a name='9-2'></a>### 9.2 - Inference PipelineCreate a simple inference function for deploying the model.

In [ ]:
def few_shot_predict(    model,    support_images,    support_labels,    query_images,    n_way):    """    Make few-shot predictions on new data.    Args:        model: trained PrototypicalNetwork        support_images: [n_way * k_shot, C, H, W] tensor        support_labels: [n_way * k_shot] tensor        query_images: [n_queries, C, H, W] tensor        n_way: number of classes    Returns:        predictions: [n_queries] predicted class labels        confidences: [n_queries] prediction confidences    """    model.eval()    with torch.no_grad():        # Move to device        support_images = support_images.to(device)        support_labels = support_labels.to(device)        query_images = query_images.to(device)        # Get logits        logits = model(support_images, support_labels, query_images, n_way)        # Get predictions and confidences        probs = F.softmax(logits, dim=1)        confidences, predictions = probs.max(dim=1)    return predictions.cpu(), confidences.cpu()# Test inferenceprint("🧪 Testing inference pipeline...\n")# Sample a test episodetest_sampler = OmniglotNWayKShot(omniglot_test, n_way=5, k_shot=1, q_query=5)support_x, support_y, query_x, query_y = test_sampler.sample_episode()# Make predictionspredictions, confidences = few_shot_predict(    model, support_x, support_y, query_x, n_way=5)# Print resultsprint("Predictions:")for i in range(len(predictions)):    true_label = query_y[i].item()    pred_label = predictions[i].item()    confidence = confidences[i].item()    correct = "✅" if pred_label == true_label else "❌"    print(f"  Query {i+1}: Predicted={pred_label}, True={true_label}, "          f"Confidence={confidence:.3f} {correct}")accuracy = (predictions == query_y).float().mean().item()print(f"\nAccuracy: {accuracy*100:.1f}%")

<a name='10'></a>## 10 - Reflection and Next Steps### 🎓 What You've Accomplished:✅ **Data Pipeline**: Loaded and sampled Omniglot episodically✅ **Model Implementation**: Built Prototypical Networks from scratch✅ **Training**: Trained with proper episodic protocol✅ **Evaluation**: Evaluated with confidence intervals✅ **Analysis**: Compared with published benchmarks✅ **Deployment**: Saved model and created inference pipeline### 📊 Reflection Questions:**1. How do your results compare to papers?**- Are you within 2-5% of published results?- If lower, what might explain the gap?  - Training episodes (2000 vs papers use 10,000+)  - Hyperparameters (learning rate, embedding dim)  - Implementation details (distance metric, normalization)**2. Which scenario was hardest? Why?**- Typically 20-way 1-shot is hardest- More classes = harder discrimination- Fewer shots = less information per class**3. What did you learn about few-shot learning?**- Importance of good embeddings- Role of prototypes as class representatives- Trade-off between N-way and K-shot**4. Where would you apply this?**- Medical imaging: Few examples of rare diseases- Robotics: Quick adaptation to new objects- Personalization: User-specific models- Low-resource languages: Translation with few examples### 🚀 Next Steps to Improve:**1. Train Longer**- Papers use 10,000-60,000 episodes- You trained for 2,000- Try 10,000+ episodes for better results**2. Hyperparameter Tuning**- Learning rate: try [1e-4, 3e-4, 1e-3, 3e-3]- Embedding dimension: try [32, 64, 128, 256]- Architecture depth: add/remove conv blocks**3. Advanced Techniques**- **Data augmentation**: rotations, noise, crops- **Learning rate scheduling**: cosine annealing- **Different distances**: Cosine, Mahalanobis- **Attention mechanisms**: weight prototypes**4. Other Algorithms**- Implement Matching Networks (Tutorial 03c)- Implement MAML (Tutorial 04)- Compare all three on same data**5. Other Datasets**- **Mini-ImageNet**: More challenging (224x224 color images)- **CUB-200**: Fine-grained bird classification- **tieredImageNet**: Hierarchical structure- Your own domain!### 📚 Recommended Papers to Read Next:1. **Prototypical Networks**: [Snell et al., 2017](https://arxiv.org/abs/1703.05175)2. **Matching Networks**: [Vinyals et al., 2016](https://arxiv.org/abs/1606.04080)3. **MAML**: [Finn et al., 2017](https://arxiv.org/abs/1703.03400)4. **Meta-Dataset**: [Triantafillou et al., 2020](https://arxiv.org/abs/1903.03096)5. **TADAM**: [Oreshkin et al., 2018](https://arxiv.org/abs/1805.10123) - Task-dependent adaptive metric### 🛠️ Code Improvements:1. **Modularize**: Separate data, models, training, eval into files2. **Config files**: Use YAML/JSON for hyperparameters3. **Logging**: Use tensorboard or wandb4. **Testing**: Unit tests for sampler, model5. **Documentation**: Docstrings, README### 🌟 Real-World Deployment:**If deploying to production:**1. **Validation set**: Use held-out validation for hyperparameter tuning2. **Model selection**: Save best model based on validation3. **Monitoring**: Track accuracy distribution, confidence calibration4. **A/B testing**: Compare to baseline in production5. **Continuous learning**: Update model as new data arrives### 💡 Challenge Yourself:**Beginner:**- [ ] Train to match paper results (within ±2%)- [ ] Visualize embeddings with t-SNE- [ ] Try different hyperparameters**Intermediate:**- [ ] Implement second algorithm (Matching/MAML)- [ ] Test on Mini-ImageNet- [ ] Add data augmentation- [ ] Implement learning rate scheduling**Advanced:**- [ ] Combine multiple algorithms (ensemble)- [ ] Implement transductive inference- [ ] Apply to your own dataset- [ ] Reproduce results from a recent paper (2023-2024)---## 🎉 CONGRATULATIONS!**You've completed the Meta-Learning course!** 🎓You now have:- ✅ Deep understanding of meta-learning principles- ✅ Hands-on experience with state-of-the-art algorithms- ✅ A working few-shot learning system- ✅ Skills to apply to real-world problems**What's Next?**1. Share your results with the community2. Apply to your own domain3. Contribute to open-source projects4. Keep learning from latest papers5. Build something amazing!**Thank you for completing this course!** 🙏We hope you found it valuable and that you'll apply these skills to solve important problems.**Happy meta-learning!** 🧠🚀---*Course created with ❤️ for the AI community**For questions, suggestions, or collaboration: [GitHub Issues](https://github.com/your-repo)*